In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi

In [ ]:
import os
WORKDIR = "/kaggle/working/AI_Lab"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
!nvcc --version


In [ ]:
!nvcc --version

## Tunnel launch

`TOKEN` and the ngrok authtoken below are placeholders substituted with the real values at push time (see `wake_kaggle()` in `src/dashboard/app.py`) -- nothing sensitive is committed to this notebook file.

In [ ]:
!pip install -q jupyter_http_over_ws pyngrok
!jupyter serverextension enable --py jupyter_http_over_ws


In [ ]:
import subprocess, time
from pyngrok import ngrok

# These two placeholders are substituted with the real values at push time
# by wake_kaggle() in the dashboard app -- never committed as real secrets.
TOKEN = "__JUPYTER_TOKEN_PLACEHOLDER__"
ngrok.set_auth_token("__NGROK_AUTHTOKEN_PLACEHOLDER__")

PORT = 8890


def launch_jupyter():
    """Starts (or restarts) the Jupyter server as a background process.
    Returns the Popen handle so the keep-alive loop can check later
    whether it's still running (.poll() is None) or has died (an exit
    code) -- wrapped in a function, not just a one-off call, so the
    keep-alive loop can relaunch it later using the exact same command."""
    return subprocess.Popen([
        "jupyter", "notebook",
        "--NotebookApp.allow_origin=*",
        "--ip=0.0.0.0",
        f"--port={PORT}",
        "--NotebookApp.port_retries=0",
        "--no-browser",
        "--allow-root",
        f"--NotebookApp.token={TOKEN}",
        f"--notebook-dir={WORKDIR}",
    ])


JUPYTER_PROC = launch_jupyter()
time.sleep(5)
print("Jupyter server launched on port", PORT, "rooted at", WORKDIR)


In [ ]:
def launch_tunnel():
    """Opens (or reopens) the ngrok tunnel to the Jupyter server -- wrapped
    in a function so the keep-alive loop can re-establish it later using
    the exact same call if it dies mid-session. Returns the tunnel object;
    .public_url is the reachable address."""
    return ngrok.connect(PORT, "http")


public_url = launch_tunnel()
print(public_url)
print(TOKEN)


## Keep-alive (activity-based, not a flat timer)

Every real dashboard action (a page-load status check, a Bond message) touches `last_activity.txt` in `WORKDIR` via the same kernel-execution bridge the dashboard already uses. This loop just watches that file and lets the run end on its own once nobody's actually used it for a while — same as closing VS Code/Jupyter and letting a session idle out, not an artificial multi-hour reservation.

In [ ]:
import time, os, json, base64, urllib.request

ACTIVITY_FILE = os.path.join(WORKDIR, "last_activity.txt")
with open(ACTIVITY_FILE, "w") as f:
    f.write(str(time.time()))

IDLE_TIMEOUT_SECONDS = 20 * 60   # end the run after 20 min with no dashboard activity
CHECK_INTERVAL_SECONDS = 30

# --- GPU-hour usage heartbeat (CLAUDE_CODE_HANDOFF_3.md) -------------------
# Records real observed runtime to data/gpu_usage_log.json in the GitHub repo
# every ~10th tick (~5 min), so the dashboard's "GPU-hour budget" card can
# show a real number instead of a static 0.0. This is this app's own
# *observed* estimate, not Kaggle's official quota meter (Kaggle exposes no
# API for that) -- deliberately never claimed as anything more. A missing or
# failing GITHUB_TOKEN must never crash the keep-alive loop or take down the
# tunnel -- every heartbeat call is best-effort, wrapped in try/except.
GITHUB_TOKEN = "__GITHUB_TOKEN_PLACEHOLDER__"
GITHUB_REPO = "fadykhella-maker/nvidia-cuda-mi-intelligent-command-center"
GPU_LOG_PATH = "data/gpu_usage_log.json"
GPU_HEARTBEAT_EVERY_N_TICKS = 10
RUN_START_TS = time.time()


def _gh_request(method, url, body=None):
    req = urllib.request.Request(url, method=method)
    req.add_header("Authorization", f"Bearer {GITHUB_TOKEN}")
    req.add_header("Accept", "application/vnd.github+json")
    data = json.dumps(body).encode() if body else None
    with urllib.request.urlopen(req, data=data, timeout=15) as r:
        return json.loads(r.read())


def record_gpu_heartbeat(run_start_ts):
    if not GITHUB_TOKEN or GITHUB_TOKEN == "__GITHUB_TOKEN_PLACEHOLDER__":
        return  # not configured -- GPU-hour tracking is optional, wake/tunnel still work fine
    api = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{GPU_LOG_PATH}"
    now = time.time()
    try:
        current = _gh_request("GET", api)
        log = json.loads(base64.b64decode(current["content"]))
        sha = current["sha"]
    except Exception:
        log, sha = {"intervals": []}, None
    intervals = log.get("intervals", [])
    if intervals and (now - intervals[-1]["end"]) < 600:  # same continuous run
        intervals[-1]["end"] = now
    else:
        intervals.append({"start": run_start_ts, "end": now})
    cutoff = now - 7 * 86400
    intervals = [iv for iv in intervals if iv["end"] >= cutoff]
    log["intervals"] = intervals
    log["updated_at"] = now
    body = {
        "message": "chore: gpu usage heartbeat",
        "content": base64.b64encode(json.dumps(log).encode()).decode(),
    }
    if sha:
        body["sha"] = sha
    _gh_request("PUT", api, body)


# --- GPU telemetry (Grafana Cloud) ------------------------------------------
# Real DCGM-named metrics, read via pynvml (DCGM itself isn't installable on
# Kaggle's containers -- see the dashboard's own Telemetry Field Schema
# table, which already promises exactly these five field names so nothing
# has to be renamed on either side). Not wired to a real push yet -- there's
# no Grafana Cloud remote-write/OTLP endpoint or credential configured here.
# GRAFANA_TELEMETRY_CONFIGURED stays False (and push_dcgm_metrics() a no-op)
# until that's added; same best-effort, never-crash-the-loop discipline as
# record_gpu_heartbeat() above.
GRAFANA_TELEMETRY_CONFIGURED = False  # flip once a real push endpoint is wired in


def read_dcgm_metrics():
    """(name, labels, value) for every GPU, named to match NVIDIA's own DCGM
    fields -- the same names the stock Grafana NVIDIA DCGM Exporter
    Dashboard already queries for, so nothing needs a rename once this is
    actually pushed somewhere."""
    import pynvml

    pynvml.nvmlInit()
    metrics = []
    try:
        for i in range(pynvml.nvmlDeviceGetCount()):
            h = pynvml.nvmlDeviceGetHandleByIndex(i)
            util = pynvml.nvmlDeviceGetUtilizationRates(h)
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            temp = pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)
            power_mw = pynvml.nvmlDeviceGetPowerUsage(h)  # milliwatts
            sm_clock = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
            labels = {"gpu": str(i)}
            metrics += [
                ("DCGM_FI_DEV_GPU_UTIL", labels, float(util.gpu)),
                ("DCGM_FI_DEV_FB_USED", labels, float(mem.used) / (1024 * 1024)),  # MiB
                ("DCGM_FI_DEV_GPU_TEMP", labels, float(temp)),
                ("DCGM_FI_DEV_POWER_USAGE", labels, power_mw / 1000.0),  # watts
                ("DCGM_FI_DEV_SM_CLOCK", labels, float(sm_clock)),  # MHz
            ]
    finally:
        pynvml.nvmlShutdown()
    return metrics


def push_dcgm_metrics():
    """Best-effort, no-op until GRAFANA_TELEMETRY_CONFIGURED is actually
    True -- deliberately not guessing at a remote-write/OTLP call without
    real Cloud Portal credentials in hand."""
    if not GRAFANA_TELEMETRY_CONFIGURED:
        return
    for name, labels, value in read_dcgm_metrics():
        pass  # TODO: replace with the real push call once wired in


# --- Tunnel/server self-heal ------------------------------------------------
# The loop below used to only ever watch last_activity.txt -- it had no idea
# whether the actual jupyter process or ngrok tunnel had silently died
# underneath it. Observed directly: Kaggle's own run status stayed "running"
# while the tunnel itself was unreachable -- this closes that gap by
# checking both are genuinely still alive every tick, and relaunching
# whichever one died, using the exact same launch_jupyter()/launch_tunnel()
# functions defined earlier rather than duplicating that logic here.
def _tunnel_alive():
    try:
        return any(t.public_url == public_url.public_url for t in ngrok.get_tunnels())
    except Exception:
        return False


def _ensure_healthy():
    global JUPYTER_PROC, public_url
    if JUPYTER_PROC.poll() is not None:
        print(f"Jupyter process died (exit {JUPYTER_PROC.poll()}) -- relaunching.")
        JUPYTER_PROC = launch_jupyter()
        time.sleep(5)
    if not _tunnel_alive():
        print("ngrok tunnel is down -- relaunching.")
        try:
            ngrok.disconnect(public_url.public_url)
        except Exception:
            pass
        public_url = launch_tunnel()
        print("New tunnel URL:", public_url)


print(f"Holding session open -- will end automatically after "
      f"{IDLE_TIMEOUT_SECONDS // 60} min with no dashboard activity.")
_tick = 0
while True:
    time.sleep(CHECK_INTERVAL_SECONDS)
    _tick += 1
    try:
        with open(ACTIVITY_FILE) as f:
            last = float(f.read().strip())
    except Exception:
        last = 0
    idle_for = time.time() - last
    if idle_for > IDLE_TIMEOUT_SECONDS:
        print(f"Idle for {idle_for:.0f}s (> {IDLE_TIMEOUT_SECONDS}s) -- ending session.")
        break
    try:
        _ensure_healthy()
    except Exception as e:
        print(f"Health check failed (non-fatal): {e}")
    if _tick % GPU_HEARTBEAT_EVERY_N_TICKS == 0:
        try:
            record_gpu_heartbeat(RUN_START_TS)
        except Exception as e:
            print(f"GPU-hour heartbeat failed (non-fatal): {e}")
    try:
        push_dcgm_metrics()
    except Exception as e:
        print(f"GPU telemetry push failed (non-fatal): {e}")
    print(f"Still active ({idle_for:.0f}s since last dashboard activity)")
